# Macro Data Exploration

Download and explore economy data from Polygon/Massive REST API.

Endpoints:
- `/fed/v1/treasury-yields` — daily yield curve (1Y, 5Y, 10Y)
- `/fed/v1/inflation` — monthly CPI
- `/fed/v1/inflation-expectations` — monthly model-based expectations
- `/fed/v1/labor-market` — monthly unemployment + participation

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.getenv('POLYGON_S3_SECRET_KEY')
BASE    = 'https://api.polygon.io'

def fetch_all(path, params=None):
    """Fetch all pages from a paginated endpoint."""
    params  = params or {}
    headers = {'Authorization': f'Bearer {API_KEY}'}
    results = []
    url     = f'{BASE}{path}'
    params['limit'] = 50000
    while url:
        r = requests.get(url, headers=headers, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        results.extend(data.get('results', []))
        url    = data.get('next_url')
        params = {}  # cursor already embedded in next_url
        print(f'  fetched {len(results)} rows...', end='\r')
    print()
    return results

print('Ready. API key loaded:', bool(API_KEY))

## 1. Treasury Yields

In [ ]:
print('Fetching treasury yields...')
raw = fetch_all('/fed/v1/treasury-yields')
df_yields = pd.DataFrame(raw)
df_yields['date'] = pd.to_datetime(df_yields['date'])
df_yields = df_yields.set_index('date').sort_index()

print(f'Shape: {df_yields.shape}')
print(f'Date range: {df_yields.index.min()} to {df_yields.index.max()}')
print(f'Columns: {df_yields.columns.tolist()}')
print()
print(df_yields.tail(10))

In [ ]:
# Compute derived features
df_yields['spread_10y_1y'] = df_yields['yield_10_year'] - df_yields['yield_1_year']  # curve slope
df_yields['spread_5y_1y']  = df_yields['yield_5_year']  - df_yields['yield_1_year']
df_yields['yield_10y_mom_20d'] = df_yields['yield_10_year'].diff(20)   # 1-month momentum
df_yields['yield_10y_mom_60d'] = df_yields['yield_10_year'].diff(60)   # 3-month momentum
df_yields['curve_mom_20d']     = df_yields['spread_10y_1y'].diff(20)

print('Last 5 rows with derived features:')
print(df_yields[['yield_1_year','yield_5_year','yield_10_year',
                  'spread_10y_1y','yield_10y_mom_20d','curve_mom_20d']].tail(5))

## 2. Inflation

In [ ]:
print('Fetching inflation...')
raw = fetch_all('/fed/v1/inflation')
df_cpi = pd.DataFrame(raw)
df_cpi['date'] = pd.to_datetime(df_cpi['date'])
df_cpi = df_cpi.set_index('date').sort_index()

print(f'Shape: {df_cpi.shape}')
print(f'Date range: {df_cpi.index.min()} to {df_cpi.index.max()}')
print(f'Columns: {df_cpi.columns.tolist()}')
print()
print(df_cpi.tail(10))

## 3. Inflation Expectations

In [ ]:
print('Fetching inflation expectations...')
raw = fetch_all('/fed/v1/inflation-expectations')
df_exp = pd.DataFrame(raw)
df_exp['date'] = pd.to_datetime(df_exp['date'])
df_exp = df_exp.set_index('date').sort_index()

print(f'Shape: {df_exp.shape}')
print(f'Date range: {df_exp.index.min()} to {df_exp.index.max()}')
print(f'Columns: {df_exp.columns.tolist()}')
print()
print(df_exp.tail(10))

## 4. Labor Market

In [ ]:
print('Fetching labor market...')
raw = fetch_all('/fed/v1/labor-market')
df_labor = pd.DataFrame(raw)
df_labor['date'] = pd.to_datetime(df_labor['date'])
df_labor = df_labor.set_index('date').sort_index()

print(f'Shape: {df_labor.shape}')
print(f'Date range: {df_labor.index.min()} to {df_labor.index.max()}')
print(f'Columns: {df_labor.columns.tolist()}')
print()
print(df_labor.tail(10))

## 5. Combine into USD Macro Feature Set

In [ ]:
# Resample everything to daily, forward-fill (monthly data fills each day)
daily_idx = pd.date_range('2009-01-01', '2025-12-31', freq='D')

macro = pd.DataFrame(index=daily_idx)

# Yields — already daily
for col in ['yield_1_year', 'yield_5_year', 'yield_10_year',
            'spread_10y_1y', 'yield_10y_mom_20d', 'yield_10y_mom_60d', 'curve_mom_20d']:
    if col in df_yields.columns:
        macro[col] = df_yields[col].reindex(daily_idx).ffill()

# CPI — monthly, forward fill
for col in df_cpi.columns:
    macro[f'cpi_{col}'] = df_cpi[col].reindex(daily_idx).ffill()

# CPI momentum
if 'cpi_cpi' in macro.columns:
    macro['cpi_yoy']   = macro['cpi_cpi'].pct_change(365) * 100
    macro['cpi_mom3m'] = macro['cpi_cpi'].pct_change(90)  * 100

# Inflation expectations — monthly
for col in df_exp.columns:
    macro[f'exp_{col}'] = df_exp[col].reindex(daily_idx).ffill()

# Real rate = 10Y yield - 1Y inflation expectation
if 'yield_10_year' in macro.columns and 'exp_model_1_year' in macro.columns:
    macro['real_rate_10y'] = macro['yield_10_year'] - macro['exp_model_1_year']

# Labor — monthly
for col in df_labor.columns:
    macro[f'labor_{col}'] = df_labor[col].reindex(daily_idx).ffill()

# Unemployment trend
if 'labor_unemployment_rate' in macro.columns:
    macro['unemp_mom3m'] = macro['labor_unemployment_rate'].diff(90)

macro = macro.dropna(how='all')
print(f'Macro feature set: {macro.shape}')
print(f'Date range: {macro.index.min()} to {macro.index.max()}')
print(f'\nColumns:')
for c in macro.columns:
    print(f'  {c:<35} last={macro[c].dropna().iloc[-1]:.4f}')

## 6. Quick Correlation with EURUSD Direction

In [ ]:
# Does any macro feature correlate with EURUSD 4H forward direction?
df_1h = pd.read_parquet('../backend/data/processed/EURUSD_1H.parquet')
df_1h = df_1h[df_1h.index <= '2024-06-30']

c = df_1h['close']
fwd_4h = np.log(c.shift(-4) / c)

# Align macro to 1H bars (daily macro -> each 1H bar gets that day's macro value)
macro_1h = macro.reindex(df_1h.index.normalize()).set_index(df_1h.index)

print(f'{"Feature":<35} {"Corr_4H":>10}')
print('=' * 48)
results = []
for col in macro.columns:
    x   = macro_1h[col]
    valid = pd.DataFrame({'x': x, 'fwd': fwd_4h}).dropna()
    if len(valid) < 1000: continue
    corr = np.corrcoef(valid['x'].values, valid['fwd'].values)[0, 1]
    results.append((col, corr))

for col, corr in sorted(results, key=lambda x: abs(x[1]), reverse=True):
    flag = ' ***' if abs(corr) > 0.02 else (' *' if abs(corr) > 0.01 else '')
    print(f'{col:<35} {corr:>+10.4f}{flag}')